In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from datetime import datetime
import joblib


In [ ]:

# Step 1: Load Dataset
# Assuming `df_modified` is your dataset with QoS and user information.
# df_modified = pd.read_csv('your_dataset.csv') # Uncomment if loading from a file

# Sample mock data structure if dataset isn't loaded (replace this with your actual data if needed)
# df_modified = pd.DataFrame(...)

# Step 2: Exploratory Data Analysis (EDA)
print("Exploratory Data Analysis:\n")

# Summary statistics for numerical features
print("\nSummary Statistics for Numerical Columns:\n", df_modified.describe().T)

# Frequency counts for categorical columns
categorical_cols = ['Service_Group', 'Service_Name', 'Device_Type']
for col in categorical_cols:
    print(f"\nFrequency count for {col}:\n", df_modified[col].value_counts())

# Distribution Analysis Visualizations

# Count of Device Types
plt.figure(figsize=(8, 5))
sns.countplot(data=df_modified, x='Device_Type', palette='viridis')
plt.title('Distribution of Device Types')
plt.xlabel('Device Type')
plt.ylabel('Count')
plt.show()

# Count of Application Types
plt.figure(figsize=(8, 5))
sns.countplot(data=df_modified, x='Service_Name', palette='plasma')
plt.title('Distribution of Application Types')
plt.xlabel('Application Type')
plt.ylabel('Count')
plt.show()

# Signal Strength Distribution per Application Type
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_modified, x='Service_Name', y='Signal_Strength', palette='coolwarm')
plt.title('Signal Strength Distribution per Application Type')
plt.xlabel('Application Type')
plt.ylabel('Signal Strength (dBm)')
plt.show()

# Latency Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df_modified['Latency'], bins=30, kde=True, color='skyblue')
plt.title('Latency Distribution')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.show()

# Bandwidth Usage Distribution per Service Group
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_modified, x='Service_Group', y='Bandwidth_Speed_per_sec_Mbps', palette='muted')
plt.title('Bandwidth Usage per Service Group')
plt.xlabel('Service Group')
plt.ylabel('Bandwidth Speed (Mbps)')
plt.show()

# Packet Loss Rate vs Priority Score
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_modified, x='Packet_Loss_Rate', y='Priority_Score', hue='Service_Group', palette='cool')
plt.title('Packet Loss Rate vs Priority Score')
plt.xlabel('Packet Loss Rate')
plt.ylabel('Priority Score')
plt.legend(title='Service Group')
plt.show()

# Buffer Occupancy Distribution by Time of Day
df_modified['hour'] = df_modified['Timestamp'].dt.hour  # Extract hour from timestamp
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_modified, x='hour', y='Buffer_Occupancy', palette='Set2')
plt.title('Buffer Occupancy by Time of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Buffer Occupancy')
plt.show()

# Correlation Heatmap for Numerical Features
plt.figure(figsize=(10, 8))
sns.heatmap(df_modified.corr(numeric_only=True), annot=True, cmap='viridis', fmt='.2f')
plt.title('Correlation Heatmap of QoS Features')
plt.show()

# Step 3: Statistical Analysis
print("Statistical Analysis:\n")
print("Correlation matrix:\n", df_modified.corr())

# Step 4: Data Preprocessing

# Extract day_of_week from Timestamp (if not already extracted)
df_modified['day_of_week'] = df_modified['Timestamp'].dt.day_name()

# Check for missing values
print("\nMissing values before handling:\n", df_modified.isnull().sum())

# Handle missing values
# Fill numerical columns with median and categorical columns with mode
numerical_cols = ['Signal_Strength', 'Packet_Loss_Rate', 'Latency', 'Jitter_ms',
                  'Bandwidth_Speed_per_sec_Mbps', 'Buffer_Occupancy', 'Usage_Minutes', 'Usage_Percentage']
for col in numerical_cols:
    df_modified[col].fillna(df_modified[col].median(), inplace=True)

for col in categorical_cols + ['day_of_week']:
    df_modified[col].fillna(df_modified[col].mode()[0], inplace=True)

# Remove duplicates (excluding Hashid for uniqueness)
df_modified = df_modified.drop_duplicates(subset=[col for col in df_modified.columns if col != 'Hashid'])

print("\nMissing values after handling:\n", df_modified.isnull().sum())
print("Number of rows after removing duplicates:", df_modified.shape[0])

# Label Encoding for categorical features
label_encoders = {}
for col in categorical_cols + ['day_of_week']:
    le = LabelEncoder()
    df_modified[col] = le.fit_transform(df_modified[col])
    label_encoders[col] = le

# Standardize numerical columns
scaler = StandardScaler()
df_modified[numerical_cols + ['hour']] = scaler.fit_transform(df_modified[numerical_cols + ['hour']])

# Define features (X) and target (y) with Hashid for tracking
X = df_modified.drop(columns=['Hashid', 'Timestamp', 'Priority_Score'])  # Drop unnecessary columns and target column
y = np.random.choice(['Low', 'Medium', 'High'], size=df_modified.shape[0])  # Mock Priority_Score for ordinal target

# Train-test split with Hashid tracking
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Separate Hashid for tracking
train_hashid = X_train['Hashid']
test_hashid = X_test['Hashid']

# Drop Hashid from the feature set before model training
X_train = X_train.drop(columns=['Hashid'])
X_test = X_test.drop(columns=['Hashid'])

# Step 5: Ordinal Classification Model with Random Forest and Hyperparameter Tuning

# Define the parameter grid for optimization
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize the Random Forest with GridSearch for hyperparameter tuning
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# Retrieve the best model
best_rf = grid_search.best_estimator_
print("\nBest Parameters:", grid_search.best_params_)

# Step 6: Evaluation on Test Set
y_pred = best_rf.predict(X_test)

# Attach Hashid to the predictions to map results back to each user/device
predictions_with_hashid = pd.DataFrame({
    'Hashid': test_hashid,
    'Actual_Priority': y_test,
    'Predicted_Priority': y_pred
})

# Display prediction results with Hashid for tracking
print("\nPredictions with Hashid Tracking:\n", predictions_with_hashid.head())

# Classification Report and Confusion Matrix
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Step 7: Feature Importance Visualization
importances = best_rf.feature_importances_
features = X_train.columns
indices = np.argsort(importances)

plt.figure(figsize=(10, 8))
plt.title('Feature Importance')
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

# Step 8: Save the Model and Encoders if Needed
# Save model
joblib.dump(best_rf, 'best_rf_model.joblib')
# Save label encoders and scaler for future use
for col, encoder in label_encoders.items():
    joblib.dump(encoder, f'label_encoder_{col}.joblib')
joblib.dump(scaler, 'scaler.joblib')
